# [과제 정답] MCP vs RAG 구조 비교 실습 — PydanticAI 기반

이 노트북은 MCP(Tool 호출)와 RAG(문서 검색) 두 방식의 **완성된 정답 코드**입니다.
같은 질문에 대해 두 방식의 답변을 비교하고, 각 아키텍처의 특성을 이해합니다.

## 환경 설정

필요한 라이브러리를 임포트하고 API 키를 로드합니다.

In [ ]:
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pydantic_ai import Agent

# .env 파일에서 OPENAI_API_KEY 로드
load_dotenv()

# 임베딩 API 호출용 클라이언트
openai_client = OpenAI()

## Part 1: MCP(Tool 호출) 방식

Agent를 생성하고 `@tool_plain` 데코레이터로 보증 정보 조회 Tool을 등록합니다.
LLM은 질문을 분석하여 이 Tool을 자동으로 호출합니다.
제품 데이터는 `../src/data/products.json` 파일에서 로드합니다 (hardcoded dict 대신 파일 기반 데이터소스).

In [ ]:
import json
from pathlib import Path

# 제품 데이터를 JSON 파일에서 로드
_DATA = json.loads(Path("../src/data/products.json").read_text(encoding="utf-8"))

# Agent 생성 — instructions로 역할 정의
mcp_agent = Agent(
    "openai:gpt-5.4",
    instructions=(
        "당신은 TechShop의 고객 상담 에이전트입니다. "
        "Tool을 활용하여 정확한 정보를 제공하세요."
    ),
)


# @tool_plain: docstring이 LLM의 Tool 선택 기준이 된다
@mcp_agent.tool_plain
def get_product_warranty(product_name: str) -> str:
    """제품 보증 정보를 조회한다. 보증 기간, 보증 범위, 제외 항목, 서비스센터 정보를 반환한다.
    고객이 보증, AS, 수리에 대해 질문할 때 사용한다."""
    warranty = _DATA["warranties"].get(product_name)
    if warranty:
        print(f"  [Tool] get_product_warranty('{product_name}') → 보증 정보 조회 완료")
        return warranty
    return f"'{product_name}'에 대한 보증 정보를 찾을 수 없습니다. 정확한 제품명을 확인해주세요."

In [ ]:
# async def — 내부에서 await를 사용하므로 비동기 함수로 정의
async def run_mcp(question: str) -> str:
    """MCP 방식으로 질문에 답변한다."""
    print(f"\n[MCP] 질문: {question}")
    # await agent.run() — Jupyter에서는 run_sync() 대신 사용
    result = await mcp_agent.run(question)
    print(f"[MCP 답변] {result.output}")
    return result.output

## Part 2: RAG 방식

문서를 임베딩하여 벡터 DB 역할을 하는 numpy 배열을 생성하고,
질문과 유사한 청크를 검색하여 LLM에게 context로 제공합니다.

In [ ]:
# 검색 대상 문서 디렉토리 (solution에서는 ../src/ 경로 참조)
DOCS_DIR = Path("../src/sample_docs")

In [ ]:
def load_and_chunk() -> list[dict]:
    """문서를 로드하고 단락 기준으로 청크 분할한다."""
    chunks = []
    for md_file in sorted(DOCS_DIR.glob("*.md")):
        content = md_file.read_text(encoding="utf-8")
        current_chunk = ""
        # 빈 줄(\n\n) 기준으로 단락 분리
        for para in content.split("\n\n"):
            para = para.strip()
            if not para:
                continue
            # 500자 이하이면 현재 청크에 병합
            if len(current_chunk) + len(para) < 500:
                current_chunk += ("\n\n" + para) if current_chunk else para
            else:
                # 크기 초과 시 현재 청크 저장 후 새 청크 시작
                if current_chunk:
                    chunks.append({"text": current_chunk, "source": md_file.name})
                current_chunk = para
        # 마지막 청크 저장
        if current_chunk:
            chunks.append({"text": current_chunk, "source": md_file.name})
    print(f"[RAG] {len(chunks)}개 청크 생성")
    return chunks

In [ ]:
def embed(texts: list[str]) -> np.ndarray:
    """텍스트를 임베딩 벡터로 변환한다. (text-embedding-3-small 사용)"""
    # OpenAI Embedding API — 여러 텍스트를 한번에 배치 임베딩
    response = openai_client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in response.data])

In [ ]:
def search(query: str, chunks: list[dict], chunk_embeddings: np.ndarray, top_k: int = 3) -> list[dict]:
    """질문과 유사한 청크를 코사인 유사도로 검색한다."""
    # 질문을 임베딩 벡터로 변환
    query_emb = embed([query])[0]
    # L2 정규화 — 코사인 유사도 계산 준비
    q_norm = query_emb / np.linalg.norm(query_emb)
    c_norm = chunk_embeddings / np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
    # 내적 = 코사인 유사도 (정규화 후)
    similarities = c_norm @ q_norm
    # 유사도 높은 순으로 정렬 후 상위 top_k개 추출
    ranked = np.argsort(similarities)[::-1][:top_k]
    results = []
    for idx in ranked:
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(similarities[idx]),
        })
        print(f"  [검색] {chunks[idx]['source']} (유사도: {similarities[idx]:.4f})")
    return results

In [ ]:
# async def — 내부에서 await agent.run()을 사용하므로 비동기 함수로 정의
async def run_rag(question: str) -> str:
    """RAG 방식으로 질문에 답변한다."""
    print(f"\n[RAG] 질문: {question}")

    # 1. 문서 로드 + 청킹
    chunks = load_and_chunk()

    # 2. 모든 청크를 임베딩 벡터로 변환
    chunk_embeddings = embed([c["text"] for c in chunks])

    # 3. 질문과 유사한 청크 top-3 검색
    retrieved = search(question, chunks, chunk_embeddings, top_k=3)

    # 4. 검색 결과를 context 문자열로 구성
    context_parts = [f"[문서 {i+1}: {c['source']}]\n{c['text']}" for i, c in enumerate(retrieved)]
    context = "\n\n---\n\n".join(context_parts)

    # PydanticAI Agent — instructions에 검색 결과 주입 + 환각 방지 규칙
    rag_agent = Agent(
        "openai:gpt-5.4",
        instructions=(
            "당신은 TechShop의 고객 상담 에이전트입니다.\n"
            "아래 제공된 문서를 바탕으로 고객의 질문에 답변하세요.\n"
            "문서에 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 답하세요.\n\n"
            f"[참조 문서]\n{context}"
        ),
    )

    # await agent.run() — Jupyter에서는 run_sync() 대신 사용
    result = await rag_agent.run(question)
    print(f"[RAG 답변] {result.output}")
    return result.output

## Part 3: 비교 실행

같은 질문을 MCP와 RAG 두 방식으로 실행하여 답변의 차이를 비교합니다.

In [ ]:
# 비교할 질문 — MCP와 RAG 모두 동일한 질문으로 실행
question = "스마트 워치 Pro의 보증 기간이 어떻게 되나요?"

### MCP 방식 실행

Tool이 자동 호출되어 구조화된 보증 정보를 반환합니다.

In [ ]:
# run_mcp는 async 함수이므로 await로 호출
mcp_answer = await run_mcp(question)

### RAG 방식 실행

문서에서 관련 청크를 검색하여 답변을 생성합니다.

In [ ]:
# run_rag는 async 함수이므로 await로 호출
rag_answer = await run_rag(question)

## Part 4: 장단점 비교

### MCP 방식의 장점
- 실시간 데이터 접근: DB나 API에서 항상 최신 정보를 가져온다
- 정확한 구조화된 응답: Tool이 반환하는 데이터는 사실이므로 환각이 없다
- 외부 시스템 조작 가능: 단순 조회뿐 아니라 생성/수정/삭제도 가능하다

### MCP 방식의 단점
- 사전에 모든 기능을 Tool로 정의해야 함: 예상치 못한 질문에 대응 불가
- Tool 수가 많아지면 LLM의 선택 정확도 저하: 15개 이상 시 성능 문제
- 비정형 질문 처리 불가: "환불 정책 전반에 대해 설명해줘" 같은 포괄적 질문 어려움

### RAG 방식의 장점
- 비정형 지식 검색에 강함: 문서 전체에서 관련 내용을 찾아 답변
- 근거 기반 답변: 어떤 문서에서 왔는지 출처 추적 가능
- 코드 변경 없이 지식 확장: 문서만 추가하면 새로운 질문에 대응

### RAG 방식의 단점
- 실시간 데이터 불가: 임베딩 시점의 정보만 검색 가능
- 검색 품질에 의존: 청킹/임베딩 전략이 답변 품질을 결정
- 복잡한 계산이나 다단계 추론 어려움: 단순 검색+생성 구조의 한계

### 어떤 상황에서 MCP가 더 적합한가?
- 실시간 데이터 조회가 핵심인 경우 (주문 상태, 재고 확인, 계좌 잔액)
- 외부 시스템을 조작해야 하는 경우 (주문 취소, 환불 처리, 이메일 발송)
- 데이터가 자주 변경되어 문서화가 비효율적인 경우

### 어떤 상황에서 RAG가 더 적합한가?
- 방대한 문서 기반 Q&A (사내 매뉴얼, 기술 문서, FAQ)
- 데이터가 정적이고 변경 빈도가 낮은 경우
- 근거 있는 답변이 중요한 경우 (법률, 정책, 규정 안내)
- 외부 시스템 연동 없이 지식 검색만 필요한 경우